# 🤖 Medical Gesture Recognition — Bidirectional LSTM

**Dataset:** 12 medical gesture classes — PAIN, YES, NO, HELP, STOP, CHEST, HEAD, STOMACH, FEVER, WATER, DOCTOR, MEDICINE  
**Model:** Bidirectional LSTM → LayerNorm → FC classifier  
**Input shape:** `(batch, 30 frames, 63 keypoints)`  

---
### ✅ Setup steps:
1. Upload your `train.zip` to Colab using the upload cell below
2. Run all cells in order
3. Download `best_model.pth`, `gesture_model.onnx`, and `labels.json` at the end

In [ ]:
# ─────────────────────────────────────────────────
# CELL 1 — Install dependencies
# ─────────────────────────────────────────────────
!pip install torch torchvision scikit-learn numpy matplotlib seaborn torchsummary --quiet
print('✅ All packages installed')

In [ ]:
# ─────────────────────────────────────────────────
# CELL 2 — Upload train.zip from your computer
# ─────────────────────────────────────────────────
from google.colab import files
import zipfile, os

print('📂 Please upload your train.zip file...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')
print('✅ Extracted successfully')

# Auto-detect the dataset path
DATASET_ROOT = None
for root, dirs, files_list in os.walk('.'):
    if 'CHEST' in dirs or 'PAIN' in dirs:
        DATASET_ROOT = root
        break
print(f'📁 Dataset found at: {DATASET_ROOT}')

In [ ]:
# ─────────────────────────────────────────────────
# CELL 3 — Imports & Config
# ─────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import json
import time
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from torchsummary import summary
import warnings
warnings.filterwarnings('ignore')

# ── Hyperparameters (tune these if needed) ────────
GESTURES     = ['PAIN','YES','NO','HELP','STOP','CHEST',
                'HEAD','STOMACH','FEVER','WATER','DOCTOR','MEDICINE']
INPUT_SIZE   = 63        # 21 hand keypoints × 3 (x,y,z)
SEQ_LEN      = 30        # 30 frames per gesture
HIDDEN_SIZE  = 128       # LSTM hidden units
NUM_LAYERS   = 2         # stacked LSTM layers
DROPOUT      = 0.3
BATCH_SIZE   = 32
EPOCHS       = 100
LR           = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOP   = 15        # patience epochs
VAL_SPLIT    = 0.2
SEED         = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🔧 Device: {DEVICE.upper()}')
if DEVICE == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
os.makedirs('models', exist_ok=True)

In [ ]:
# ─────────────────────────────────────────────────
# CELL 4 — Dataset loader
# ─────────────────────────────────────────────────
class GestureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


def load_data(dataset_root):
    X, y = [], []
    counts = {}
    skipped = 0
    for label, gesture in enumerate(GESTURES):
        folder = os.path.join(dataset_root, gesture)
        if not os.path.exists(folder):
            print(f'⚠️  Missing folder: {folder}')
            counts[gesture] = 0
            continue
        files = sorted([f for f in os.listdir(folder) if f.endswith('.npy')])
        ok = 0
        for fname in files:
            seq = np.load(os.path.join(folder, fname))
            if seq.shape == (SEQ_LEN, INPUT_SIZE):
                X.append(seq)
                y.append(label)
                ok += 1
            else:
                skipped += 1
        counts[gesture] = ok

    print('\n📊 Data loaded:')
    print(f'  {"Gesture":<12}  {"Samples":>7}')
    print('  ' + '─' * 22)
    for g, c in counts.items():
        bar = '█' * (c // 5)
        print(f'  {g:<12}  {c:>7}  {bar}')
    print('  ' + '─' * 22)
    print(f'  {"TOTAL":<12}  {len(X):>7}')
    if skipped:
        print(f'  ⚠️  Skipped {skipped} files with wrong shape')
    return np.array(X, dtype=np.float32), np.array(y)


X, y = load_data(DATASET_ROOT)
assert len(X) > 0, '❌ No data found!'

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VAL_SPLIT, stratify=y, random_state=SEED
)
print(f'\n✅ Train: {len(X_train)} | Val: {len(X_val)}')

train_dl = DataLoader(GestureDataset(X_train, y_train),
                      batch_size=BATCH_SIZE, shuffle=True,  pin_memory=(DEVICE=='cuda'))
val_dl   = DataLoader(GestureDataset(X_val,   y_val),
                      batch_size=BATCH_SIZE, pin_memory=(DEVICE=='cuda'))

In [ ]:
# ─────────────────────────────────────────────────
# CELL 5 — Model definition
# ─────────────────────────────────────────────────
class GestureNet(nn.Module):
    """
    Bidirectional LSTM for gesture sequence classification.
    Input:  (batch, 30, 63)
    Output: (batch, num_classes)

    Architecture:
      BiLSTM(128 hidden, 2 layers, dropout=0.3)
        → LayerNorm(256)
        → Linear(256 → 64) + ReLU + Dropout(0.3)
        → Linear(64 → num_classes)
    """
    def __init__(self, num_classes=12, input_size=INPUT_SIZE,
                 hidden=HIDDEN_SIZE, layers=NUM_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Linear(hidden * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(out[:, -1, :])  # last timestep


model = GestureNet(num_classes=len(GESTURES)).to(DEVICE)

# Print model summary
print('═' * 55)
print('         GestureNet — Model Summary')
print('═' * 55)
summary(model, (SEQ_LEN, INPUT_SIZE), device=DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal params:     {total_params:,}')
print(f'Trainable params: {trainable:,}')

In [ ]:
# ─────────────────────────────────────────────────
# CELL 6 — Training loop with full graphs
# ─────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5, verbose=False
)
loss_fn = nn.CrossEntropyLoss()

# ── History trackers
history = {
    'train_loss': [], 'val_loss': [], 'val_acc': [], 'lr': []
}
best_val_acc      = 0.0
patience_counter  = 0
best_epoch        = 0
start_time        = time.time()
last_preds        = []
last_labels       = []

print(f'🚀 Starting training on {DEVICE.upper()} — {EPOCHS} max epochs\n')
print(f'{"Epoch":>6}  {"Train Loss":>11}  {"Val Loss":>9}  {"Val Acc":>9}  {"LR":>10}')
print('─' * 55)

for epoch in range(EPOCHS):
    # ── Train ────────────────────────────────────
    model.train()
    train_loss = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    # ── Validate ─────────────────────────────────
    model.eval()
    val_loss = 0.0
    correct  = 0
    preds_all, labels_all = [], []
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            val_loss += loss_fn(logits, yb).item()
            preds = logits.argmax(1)
            correct += (preds == yb).sum().item()
            preds_all.extend(preds.cpu().tolist())
            labels_all.extend(yb.cpu().tolist())

    avg_train = train_loss / len(train_dl)
    avg_val   = val_loss   / len(val_dl)
    val_acc   = correct / len(X_val) * 100
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(avg_train)
    history['val_loss'].append(avg_val)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    scheduler.step(avg_val)

    flag = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch   = epoch + 1
        patience_counter = 0
        last_preds   = preds_all
        last_labels  = labels_all
        torch.save(model.state_dict(), 'models/best_model.pth')
        flag = '  ← best ⭐'
    else:
        patience_counter += 1

    print(f'{epoch+1:>6}  {avg_train:>11.4f}  {avg_val:>9.4f}  {val_acc:>8.2f}%  {current_lr:>10.2e}{flag}')

    if patience_counter >= EARLY_STOP:
        print(f'\n⏹  Early stopping at epoch {epoch+1} (patience={EARLY_STOP})')
        break

elapsed = time.time() - start_time
print(f'\n✅ Training complete in {elapsed/60:.1f} min')
print(f'🏆 Best val acc: {best_val_acc:.2f}% at epoch {best_epoch}')

In [ ]:
# ─────────────────────────────────────────────────
# CELL 7 — 📊 Full Training Graphs
# ─────────────────────────────────────────────────
epochs_ran = len(history['train_loss'])
ep = range(1, epochs_ran + 1)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('GestureNet — Training Dashboard', fontsize=18, fontweight='bold', y=1.01)

# ── 1. Loss curve ─────────────────────────────────
ax = axes[0, 0]
ax.plot(ep, history['train_loss'], label='Train Loss', color='#4C9BE8', linewidth=2)
ax.plot(ep, history['val_loss'],   label='Val Loss',   color='#E8734C', linewidth=2)
ax.axvline(best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
ax.set_title('Loss over Epochs', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_facecolor('#f9f9f9')

# ── 2. Accuracy curve ─────────────────────────────
ax = axes[0, 1]
ax.plot(ep, history['val_acc'], color='#5DB85D', linewidth=2.5, label='Val Accuracy')
ax.axhline(best_val_acc, color='orange', linestyle='--', alpha=0.8,
           label=f'Best: {best_val_acc:.2f}%')
ax.axvline(best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
ax.fill_between(ep, history['val_acc'], alpha=0.15, color='#5DB85D')
ax.set_title('Validation Accuracy over Epochs', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 105)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_facecolor('#f9f9f9')

# ── 3. Learning rate schedule ─────────────────────
ax = axes[1, 0]
ax.plot(ep, history['lr'], color='#9B59B6', linewidth=2)
ax.set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2e'))
ax.grid(True, alpha=0.3)
ax.set_facecolor('#f9f9f9')

# ── 4. Train vs Val loss gap ─────────────────────
ax = axes[1, 1]
gap = [v - t for t, v in zip(history['train_loss'], history['val_loss'])]
colors = ['#E8734C' if g > 0 else '#4C9BE8' for g in gap]
ax.bar(ep, gap, color=colors, alpha=0.8, width=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Val Loss − Train Loss (Overfitting Gap)', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Gap')
ax.grid(True, alpha=0.3, axis='y')
ax.set_facecolor('#f9f9f9')

plt.tight_layout()
plt.savefig('models/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: models/training_curves.png')

In [ ]:
# ─────────────────────────────────────────────────
# CELL 8 — Confusion Matrix
# ─────────────────────────────────────────────────
cm = confusion_matrix(last_labels, last_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100  # row-normalized %

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle(f'Confusion Matrix — Best Model (epoch {best_epoch}, val acc {best_val_acc:.2f}%)',
             fontsize=14, fontweight='bold')

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=GESTURES, yticklabels=GESTURES,
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Raw Counts', fontsize=12)
axes[0].set_xlabel('Predicted', fontsize=11)
axes[0].set_ylabel('Actual', fontsize=11)
axes[0].tick_params(axis='x', rotation=45)

# Normalized %
sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='YlOrRd',
            xticklabels=GESTURES, yticklabels=GESTURES,
            ax=axes[1], linewidths=0.5, vmin=0, vmax=100)
axes[1].set_title('Row-Normalized (%)', fontsize=12)
axes[1].set_xlabel('Predicted', fontsize=11)
axes[1].set_ylabel('Actual', fontsize=11)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('models/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: models/confusion_matrix.png')

In [ ]:
# ─────────────────────────────────────────────────
# CELL 9 — Per-class F1 / Precision / Recall chart
# ─────────────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score

prec = precision_score(last_labels, last_preds, average=None, labels=list(range(len(GESTURES))))
rec  = recall_score   (last_labels, last_preds, average=None, labels=list(range(len(GESTURES))))
f1   = f1_score       (last_labels, last_preds, average=None, labels=list(range(len(GESTURES))))

x = np.arange(len(GESTURES))
width = 0.25

fig, ax = plt.subplots(figsize=(16, 6))
bars1 = ax.bar(x - width, prec * 100, width, label='Precision', color='#4C9BE8', alpha=0.85)
bars2 = ax.bar(x,         rec  * 100, width, label='Recall',    color='#5DB85D', alpha=0.85)
bars3 = ax.bar(x + width, f1   * 100, width, label='F1-Score',  color='#E8734C', alpha=0.85)

ax.set_title('Per-Class Precision / Recall / F1 — Best Model', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(GESTURES, rotation=30, ha='right')
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 115)
ax.legend(fontsize=12)
ax.grid(True, axis='y', alpha=0.3)
ax.set_facecolor('#f9f9f9')

# Value labels on bars
for bar in list(bars1) + list(bars2) + list(bars3):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 1.5, f'{h:.0f}',
            ha='center', va='bottom', fontsize=7, fontweight='bold')

plt.tight_layout()
plt.savefig('models/per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

# Print classification report
print('\n📋 Full Classification Report:')
print(classification_report(last_labels, last_preds, target_names=GESTURES, digits=3))
print('✅ Saved: models/per_class_metrics.png')

In [ ]:
# ─────────────────────────────────────────────────
# CELL 10 — Model Architecture Diagram
# ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 14))
ax.axis('off')
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#1a1a2e')

layers = [
    ('Input',            '(batch, 30 frames, 63 keypoints)',  '#2196F3'),
    ('BiLSTM Layer 1',   'hidden=128 × 2 (bidir) = 256',     '#9C27B0'),
    ('BiLSTM Layer 2',   'hidden=128 × 2 (bidir) = 256',     '#9C27B0'),
    ('Last Timestep',    'take output[:, -1, :]  → (B,256)', '#FF9800'),
    ('LayerNorm',        'normalize 256-dim vector',          '#607D8B'),
    ('Linear 256→64',    '+ ReLU + Dropout(0.3)',            '#4CAF50'),
    ('Linear 64→12',     '12 gesture classes',               '#F44336'),
    ('Output',           'softmax → predicted gesture',       '#00BCD4'),
]

total = len(layers)
for i, (name, detail, color) in enumerate(layers):
    y_pos = 1 - (i + 0.5) / total
    rect = plt.Rectangle((0.1, y_pos - 0.038), 0.8, 0.07,
                          color=color, alpha=0.85, zorder=2, linewidth=2,
                          edgecolor='white')
    ax.add_patch(rect)
    ax.text(0.5, y_pos + 0.01, name, ha='center', va='center',
            fontsize=13, fontweight='bold', color='white', zorder=3)
    ax.text(0.5, y_pos - 0.018, detail, ha='center', va='center',
            fontsize=9, color='white', alpha=0.85, zorder=3)
    # Arrow
    if i < total - 1:
        y_arrow = y_pos - 0.038
        ax.annotate('', xy=(0.5, y_arrow - 0.015),
                    xytext=(0.5, y_arrow + 0.001),
                    arrowprops=dict(arrowstyle='->', color='white', lw=2))

ax.set_title('GestureNet Architecture', fontsize=16, fontweight='bold',
             color='white', pad=20)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('models/model_architecture.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1a2e')
plt.show()
print('✅ Saved: models/model_architecture.png')

In [ ]:
# ─────────────────────────────────────────────────
# CELL 11 — Export ONNX + label map
# ─────────────────────────────────────────────────
model.eval()
model.load_state_dict(torch.load('models/best_model.pth', map_location=DEVICE))
model_cpu = model.cpu()

dummy = torch.zeros(1, SEQ_LEN, INPUT_SIZE)
torch.onnx.export(
    model_cpu, dummy, 'models/gesture_model.onnx',
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}},
    opset_version=12,
    verbose=False
)

label_map = {str(i): g for i, g in enumerate(GESTURES)}
with open('models/labels.json', 'w') as f:
    json.dump(label_map, f, indent=2)

# Summary
print('\n📦 Exported files:')
for fname in os.listdir('models'):
    size = os.path.getsize(f'models/{fname}')
    print(f'  models/{fname}  ({size/1024:.1f} KB)')

print(f'\n🏆 Final Results:')
print(f'   Best Val Accuracy : {best_val_acc:.2f}%')
print(f'   Best Epoch        : {best_epoch}')
print(f'   Total Params      : {total_params:,}')
print(f'   Training Time     : {elapsed/60:.1f} min on {DEVICE.upper()}')

In [ ]:
# ─────────────────────────────────────────────────
# CELL 12 — Download all output files
# ─────────────────────────────────────────────────
from google.colab import files

download_files = [
    'models/best_model.pth',
    'models/gesture_model.onnx',
    'models/labels.json',
    'models/training_curves.png',
    'models/confusion_matrix.png',
    'models/per_class_metrics.png',
    'models/model_architecture.png',
]

print('⬇️  Downloading files...')
for f in download_files:
    if os.path.exists(f):
        files.download(f)
        print(f'  ✅ {f}')
    else:
        print(f'  ⚠️  Missing: {f}')

print('\n🎉 Done! Check your Downloads folder.')